In [1]:
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
!pip install -q fastapi uvicorn nest-asyncio huggingface_hub


!wget -q -O cloudflared \
https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64

!chmod +x cloudflared

import nest_asyncio
nest_asyncio.apply()

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 GB 889.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00


In [ ]:
# === Configuration ===
# Model
MODEL_REPO = "unsloth/gemma-4-E4B-it-qat-GGUF"
MODEL_FILENAME = "gemma-4-E4B-it-qat-UD-Q4_K_XL.gguf"

# Server
LLM_HOST = "0.0.0.0"
LLM_PORT = 8000

# Model params (CPU mode — no GPU)
LLM_N_CTX = 16384
LLM_N_GPU_LAYERS = 0
LLM_N_BATCH = 512

In [ ]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILENAME
)

print(model_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


gemma-4-E4B-it-qat-UD-Q4_K_XL.gguf:   0%|          | 0.00/4.22G [00:00<?, ?B/s]

/root/.cache/huggingface/hub/models--unsloth--gemma-4-E4B-it-qat-GGUF/snapshots/bbcd9d849c2541ecc2af7ef64b3c3c2c7aa14e96/gemma-4-E4B-it-qat-UD-Q4_K_XL.gguf


In [ ]:
from llama_cpp import Llama
import time

print("Loading Gemma...")

start = time.time()

llm = Llama(
    model_path=model_path,
    n_ctx=LLM_N_CTX,
    n_gpu_layers=LLM_N_GPU_LAYERS,
    n_batch=LLM_N_BATCH,
    verbose=False
)

print(
    f"Loaded in {time.time()-start:.2f}s"
)

Loading Gemma...


llama_context: n_ctx_seq (16384) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024


Loaded in 4.79s


In [4]:
from fastapi import FastAPI
from pydantic import BaseModel
import time

app = FastAPI(
    title="Moroccan Legal RAG - Gemma4-E2B-QAT",
    version="1.0.0"
)

SYSTEM_PROMPT = (
    "انت مساعد قانوني مغربي متخصص. تجيب على الاسئلة القانونية بناء على "
    "النصوص القانونية المقدمة فقط. لا تخترع معلومات غير موجودة في النصوص. "
    "تذكر ارقام المواد التي استندت اليها في اجابتك. "
    "اذا لم تكن الاجابة موجودة في النصوص المقدمة فقل ذلك بوضوح.\n\n"
    "قدم اجابتك بهذا التنسيق:\n"
    "1. اجابة مختصرة\n"
    "2. شرح مفصل\n"
    "3. مثال عملي\n"
    "4. المواد القانونية المستند اليها"
)


class LLMRequest(BaseModel):
    prompt: str
    max_tokens: int = 4096
    temperature: float = 0.3
    top_p: float = 0.9


class LLMResponse(BaseModel):
    response: str
    tokens_generated: int
    time_ms: float



In [5]:
@app.get("/")
async def root():
    return {
        "service": "Moroccan Legal RAG - Gemma4-E2B-QAT",
        "model": MODEL_ID,
        "backend": "llama.cpp"
    }



In [6]:
@app.get("/health")
async def health():
    return {
        "status": "ok",
        "model": MODEL_ID
    }



In [7]:
@app.post("/generate", response_model=LLMResponse)
async def generate(request: LLMRequest):

    start = time.time()

    response = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": request.prompt
            }
        ],
        max_tokens=request.max_tokens,
        temperature=request.temperature,
        top_p=request.top_p,
    )

    generated_text = (
        response["choices"][0]["message"]["content"]
    )

    elapsed = (
        time.time() - start
    ) * 1000

    tokens_generated = (
        response["usage"]["completion_tokens"]
        if "usage" in response
        else len(generated_text.split())
    )

    return LLMResponse(
        response=generated_text.strip(),
        tokens_generated=tokens_generated,
        time_ms=round(elapsed, 2),
    )

In [8]:
test_prompt = """
النصوص القانونية:

[المادة 42 من مدونة التجارة]:
يجب على كل شخص ملزم بالتسجيل في السجل التجاري
ان يبين في فاتوراته ومراسلاته رقم التسجيل.

السؤال:
ما هي شروط التسجيل في السجل التجاري؟
"""

result = llm.create_chat_completion(
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": test_prompt
        }
    ],
    max_tokens=512,
    temperature=0.3,
)

print(
    result["choices"][0]["message"]["content"]
)

1. الإجابة المختصرة: النصوص القانونية المقدمة لا تحدد شروط التسجيل في السجل التجاري.
2. الشرح المفصل: النص القانوني المقدم، وهو المادة 42 من مدونة التجارة، يفرض التزاماً على كل شخص مسجل في السجل التجاري بضرورة إبداء رقم التسجيل في فواتيره ومراسلاته، ولكنه لا يذكر الشروط التي يجب توافرها ليتم التسجيل في السجل التجاري.
3. مثال عملي: لا يمكن تقديم مثال عملي حول شروط التسجيل بناءً على النص المقدم.
4. المواد القانونية المستند إليها: المادة 42 من مدونة التجارة.


In [ ]:
import uvicorn
import threading

def run_api():
    uvicorn.run(
        app,
        host=LLM_HOST,
        port=LLM_PORT
    )

threading.Thread(
    target=run_api,
    daemon=True
).start()

In [ ]:
import subprocess
import threading
import uvicorn
import re

def start_tunnel():
    time.sleep(3)
    process = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", f"http://{LLM_HOST}:{LLM_PORT}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in process.stdout:
        print(line.strip())
        match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
        if match:
            print()
            print("=" * 60)
            print("PUBLIC TUNNEL URL:")
            print(f"   {match.group(0)}")
            print("=" * 60)
            print()
            print("Add to .env:")
            print(f'  LLM_SERVICE_URL={match.group(0)}')
            print()
            print("Test:")
            print(f'  curl {match.group(0)}/health')
            print()

tunnel_thread = threading.Thread(target=start_tunnel, daemon=True)
tunnel_thread.start()

print("Starting Gemma LLM service...")
config = uvicorn.Config(app, host=LLM_HOST, port=LLM_PORT, log_level="info")
server = uvicorn.Server(config)
await server.serve()

Starting Gemma LLM service...


/usr/local/lib/python3.12/dist-packages/uvicorn/server.py:75: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
Exception in thread Thread-4 (run_api):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_1451/3902518954.py", line 5, in run_api
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/main.py", line 620, in run
    server.run()
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 75, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: _patch_asyncio.<locals>.run() got an unexpected keyword argum

2026-06-10T15:20:21Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-10T15:20:21Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-10T15:20:25Z INF +--------------------------------------------------------------------------------------------+
2026-06-10T15:20:25Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-10T15:20:25Z INF |  https://characters-ohio-inch-flexible.trycloudflare.c